# 🧪 MolE Data Construction for Self-Supervised Pretraining (Fixed)

This notebook demonstrates the core data construction pipeline for MolE's self-supervised pretraining using **Masked Language Modeling**.

## Key Concepts:
- **Atom Environment Tokenization**: Each atom represented by its chemical environment
- **Masked Language Modeling**: Randomly mask tokens and predict them
- **Self-Supervised Learning**: Learn from molecular structure without labels

**Dataset Scale**: ~842 Million molecules for pretraining

In [1]:
import pickle
import numpy as np
import torch
from collections import defaultdict

# RDKit for molecular processing
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, AllChem

print("✅ Core imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ Core imports successful!
PyTorch version: 2.7.0+cu126
NumPy version: 1.25.2
CUDA available: True


## 1. 🧬 Sample Molecules for Pretraining

In [2]:
# Sample molecules representing different chemical classes
sample_molecules = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Penicillin G": "CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C",
    "Glucose": "C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O",
    "Benzene": "C1=CC=CC=C1",
    "Ethanol": "CCO",
    "Morphine": "CN1CC[C@]23C4=C5C=CC(=O)C=C5CC[C@H]2[C@H]1C[C@H]3[C@H]4O",
    "ATP": "C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N"
}

print("📋 Sample Pretraining Molecules:")
for name, smiles in sample_molecules.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        num_atoms = mol.GetNumAtoms()
        print(f"  {name:12}: {smiles:60} ({num_atoms:2d} atoms)")
    else:
        print(f"  {name:12}: INVALID SMILES")

📋 Sample Pretraining Molecules:
  Aspirin     : CC(=O)OC1=CC=CC=C1C(=O)O                                     (13 atoms)
  Caffeine    : CN1C=NC2=C1C(=O)N(C(=O)N2C)C                                 (14 atoms)
  Penicillin G: CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C (23 atoms)
  Glucose     : C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O               (12 atoms)
  Benzene     : C1=CC=CC=C1                                                  ( 6 atoms)
  Ethanol     : CCO                                                          ( 3 atoms)
  Morphine    : CN1CC[C@]23C4=C5C=CC(=O)C=C5CC[C@H]2[C@H]1C[C@H]3[C@H]4O     (21 atoms)
  ATP         : C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N (31 atoms)


## 2. 📚 Simple Vocabulary Implementation

Let's implement a simplified vocabulary system for demonstration:

In [3]:
class SimpleVocabulary:
    """Simplified vocabulary for atom environments"""
    
    def __init__(self):
        self.token_to_id = {
            '<PAD>': 0,    # Padding token
            '<MASK>': 1,   # Mask token for MLM
            '<UNK>': 2,    # Unknown token
        }
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.next_id = 3
        
        # Special token IDs
        self.pad_token_id = 0
        self.mask_token_id = 1
        self.unk_token_id = 2
    
    def add_token(self, token):
        """Add a new token to vocabulary"""
        if token not in self.token_to_id:
            self.token_to_id[token] = self.next_id
            self.id_to_token[self.next_id] = token
            self.next_id += 1
        return self.token_to_id[token]
    
    def encode_atom_env(self, fingerprint):
        """Convert Morgan fingerprint to token ID"""
        token = str(fingerprint)  # Convert fingerprint to string token
        return self.add_token(token)
    
    def get_token_id(self, token):
        """Get token ID, return UNK if not found"""
        return self.token_to_id.get(token, self.unk_token_id)
    
    def __len__(self):
        return len(self.token_to_id)

# Try to load the pre-trained vocabulary, fallback to simple one
vocab_path = '../mole/data/vocabularies/vocabulary_207atomenvs_radius0_ZINC_guacamole.pkl'

try:
    with open(vocab_path, 'rb') as f:
        vocab_dict = pickle.load(f)
    
    # Create a wrapper for the loaded vocabulary
    class LoadedVocabulary:
        def __init__(self, vocab_dict):
            self.vocab_dict = vocab_dict
            self.mask_token_id = max(vocab_dict.values()) + 1  # Assume mask token is after vocab
            self.pad_token_id = 0
            self.unk_token_id = max(vocab_dict.values()) + 2
        
        def encode_atom_env(self, fingerprint):
            return self.vocab_dict.get(fingerprint, self.unk_token_id)
        
        def __len__(self):
            return len(self.vocab_dict) + 3  # +3 for special tokens
    
    vocabulary = LoadedVocabulary(vocab_dict)
    print(f"📚 Pre-trained vocabulary loaded!")
    print(f"   Vocabulary size: {len(vocab_dict):,} unique atom environments")
    
except FileNotFoundError:
    print(f"📚 Using simple vocabulary (pre-trained vocab not found)")
    vocabulary = SimpleVocabulary()

print(f"   Final vocabulary size: {len(vocabulary):,}")
print(f"   Mask token ID: {vocabulary.mask_token_id}")
print(f"   Pad token ID: {vocabulary.pad_token_id}")

📚 Pre-trained vocabulary loaded!
   Vocabulary size: 207 unique atom environments
   Final vocabulary size: 210
   Mask token ID: 208
   Pad token ID: 0


## 3. 🔧 Molecular Tokenization (FIXED)

Convert molecules to token sequences using Morgan fingerprints:

In [4]:
def tokenize_molecule(smiles, vocabulary, radius=0):
    """Convert SMILES to token sequence using Morgan fingerprints"""
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        raise ValueError(f"Invalid SMILES string: {smiles}")
    
    tokens = []
    atom_info = []
    

    from rdkit.Chem.rdMolDescriptors import GetMorganFingerprint
    
    info = {}
    fp = AllChem.GetMorganFingerprint(
        mol, 
        radius=radius, 
        bitInfo=info,
        includeRedundantEnvironments=True
    )
    
    # Extract atom environments from bitInfo
    for atom_idx in range(mol.GetNumAtoms()):
        atom = mol.GetAtomWithIdx(atom_idx)
        
        # Find the fingerprint bit for this atom
        atom_fingerprint = None
        for bit, atom_list in info.items():
            for atom_info_tuple in atom_list:
                if atom_info_tuple[0] == atom_idx and atom_info_tuple[1] == radius:
                    atom_fingerprint = bit
                    break
            if atom_fingerprint is not None:
                break

        # Raise error if no fingerprint found - no fallback
        if atom_fingerprint is None:
            raise RuntimeError(
                f"Failed to find Morgan fingerprint for atom {atom_idx} ({atom.GetSymbol()}) "
                f"in molecule {smiles}. This indicates a problem with the fingerprint generation."
            )
        
        token_id = vocabulary.encode_atom_env(atom_fingerprint)
        tokens.append(token_id)
        
        atom_info.append({
            'atom_idx': atom_idx,
            'symbol': atom.GetSymbol(),
            'fingerprint': atom_fingerprint,
            'token_id': token_id
        })
        

    
    return {
        'tokens': tokens,
        'atom_info': atom_info,
        'num_atoms': mol.GetNumAtoms()
    }

def demonstrate_tokenization(name, smiles, vocabulary):
    """Show detailed tokenization process"""
    print(f"\n🧬 Tokenizing: {name}")
    print(f"   SMILES: {smiles}")
    
    result = tokenize_molecule(smiles, vocabulary)
    if not result:
        print("   ❌ Invalid SMILES or tokenization failed")
        return None
    
    print(f"   Atoms: {result['num_atoms']}")
    print(f"   Tokens: {len(result['tokens'])}")
    
    print(f"\n   🔍 Atom-by-atom breakdown:")
    for info in result['atom_info']:
        print(f"     Atom {info['atom_idx']:2d}: {info['symbol']:2s} → FP: {str(info['fingerprint']):>12s} → Token: {info['token_id']:>3d}")
    
    print(f"\n   🎯 Token sequence: {result['tokens']}")
    return result

# Test the fixed tokenization
print("🔧 Testing Fixed Tokenization:")
print("=" * 40)

# Test with simple molecules first
simple_test = demonstrate_tokenization("Ethanol", sample_molecules["Ethanol"], vocabulary)
caffeine_result = demonstrate_tokenization("Caffeine", sample_molecules["Caffeine"], vocabulary)

🔧 Testing Fixed Tokenization:

🧬 Tokenizing: Ethanol
   SMILES: CCO
   Atoms: 3
   Tokens: 3

   🔍 Atom-by-atom breakdown:
     Atom  0: C  → FP:   2246728737 → Token:  13
     Atom  1: C  → FP:   2245384272 → Token:  32
     Atom  2: O  → FP:    864662311 → Token: 124

   🎯 Token sequence: [13, 32, 124]

🧬 Tokenizing: Caffeine
   SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
   Atoms: 14
   Tokens: 14

   🔍 Atom-by-atom breakdown:
     Atom  0: C  → FP:   2246728737 → Token:  13
     Atom  1: N  → FP:   2092489639 → Token: 180
     Atom  2: C  → FP:   3218693969 → Token: 144
     Atom  3: N  → FP:   2041434490 → Token: 159
     Atom  4: C  → FP:   3217380708 → Token: 150
     Atom  5: C  → FP:   3217380708 → Token: 150
     Atom  6: C  → FP:   3217380708 → Token: 150
     Atom  7: O  → FP:    864942730 → Token:  54
     Atom  8: N  → FP:   2092489639 → Token: 180
     Atom  9: C  → FP:   3217380708 → Token: 150
     Atom 10: O  → FP:    864942730 → Token:  54
     Atom 11: N  → FP:   209248963

## 4. 🎭 Masked Language Modeling

Create training data by randomly masking tokens:

In [5]:
class MaskedLanguageModelingPreprocessor:
    """Creates masked language modeling data for pretraining"""
    
    def __init__(self, vocabulary, mask_prob=0.15, replace_prob=0.8, random_prob=0.1):
        self.vocabulary = vocabulary
        self.mask_prob = mask_prob       # Probability of masking a token
        self.replace_prob = replace_prob # Of masked tokens, % to replace with [MASK]
        self.random_prob = random_prob   # Of masked tokens, % to replace with random
        # Remaining (1 - replace_prob - random_prob) keep original token
        
        self.mask_token_id = vocabulary.mask_token_id
        self.vocab_size = len(vocabulary)
    
    def create_masked_sample(self, tokens):
        """Create masked version for MLM training"""
        tokens = np.array(tokens, dtype=np.int64)
        original_tokens = tokens.copy()
        
        # Decide which positions to mask
        mask_decisions = np.random.random(len(tokens)) < self.mask_prob
        masked_positions = np.where(mask_decisions)[0]
        
        # Ensure at least one token is masked
        if len(masked_positions) == 0:
            masked_positions = [np.random.randint(0, len(tokens))]
            mask_decisions[masked_positions[0]] = True
        
        # For each masked position, decide the replacement strategy
        for pos in masked_positions:
            rand = np.random.random()
            
            if rand < self.replace_prob:
                # Replace with [MASK] token
                tokens[pos] = self.mask_token_id
            elif rand < self.replace_prob + self.random_prob:
                # Replace with random token (avoid special tokens)
                tokens[pos] = np.random.randint(3, min(self.vocab_size, 1000))
            # else: keep original token
        
        # Create labels: -100 for non-masked positions (ignored in loss)
        labels = np.full_like(original_tokens, -100, dtype=np.int64)
        labels[masked_positions] = original_tokens[masked_positions]
        
        return {
            'input_tokens': tokens.tolist(),
            'labels': labels.tolist(),
            'original_tokens': original_tokens.tolist(),
            'masked_positions': masked_positions,
            'mask_decisions': mask_decisions.tolist()
        }

# Create MLM preprocessor (only if tokenization worked)
if simple_test or caffeine_result:
    mlm_processor = MaskedLanguageModelingPreprocessor(vocabulary)
    
    print("🎭 Masked Language Modeling Preprocessor Created!")
    print(f"   Masking probability: {mlm_processor.mask_prob:.1%}")
    print(f"   [MASK] replacement: {mlm_processor.replace_prob:.1%} of masked tokens")
    print(f"   Random replacement: {mlm_processor.random_prob:.1%} of masked tokens")
    print(f"   Keep original: {1-mlm_processor.replace_prob-mlm_processor.random_prob:.1%} of masked tokens")
else:
    print("⚠️ Skipping MLM preprocessor - tokenization failed")

🎭 Masked Language Modeling Preprocessor Created!
   Masking probability: 15.0%
   [MASK] replacement: 80.0% of masked tokens
   Random replacement: 10.0% of masked tokens
   Keep original: 10.0% of masked tokens


## 5. 🎯 Masking Examples

See how masking works on real molecules:

In [6]:
def demonstrate_masking(name, tokens, mlm_processor, num_examples=3):
    """Show multiple masking examples for the same molecule"""
    print(f"\n🎭 Masking Examples for {name}:")
    print(f"   Original tokens: {tokens}")
    print(f"   Sequence length: {len(tokens)}")
    
    for i in range(num_examples):
        masked_sample = mlm_processor.create_masked_sample(tokens)
        
        print(f"\n   📝 Example {i+1}:")
        print(f"      Input:            {masked_sample['input_tokens']}")
        print(f"      Labels:           {masked_sample['labels']}")
        print(f"      Masked positions: {masked_sample['masked_positions']}")
        
        # Show changes in detail
        changes = []
        for pos in masked_sample['masked_positions']:
            original = masked_sample['original_tokens'][pos]
            input_val = masked_sample['input_tokens'][pos]
            
            if input_val == mlm_processor.mask_token_id:
                changes.append(f"pos{pos}({original}→[MASK])")
            elif input_val != original:
                changes.append(f"pos{pos}({original}→{input_val})")
            else:
                changes.append(f"pos{pos}({original}→kept)")
        
        print(f"      Changes:          {', '.join(changes)}")
        
        # Show loss calculation info
        num_targets = sum(1 for label in masked_sample['labels'] if label != -100)
        print(f"      Loss targets:     {num_targets}/{len(tokens)} positions")

# Test masking (only if we have successful tokenization)
if 'mlm_processor' in locals():
    if simple_test:
        demonstrate_masking("Ethanol", simple_test['tokens'], mlm_processor)
    
    if caffeine_result:
        demonstrate_masking("Caffeine", caffeine_result['tokens'], mlm_processor)
else:
    print("⚠️ Skipping masking examples - MLM processor not available")


🎭 Masking Examples for Ethanol:
   Original tokens: [13, 32, 124]
   Sequence length: 3

   📝 Example 1:
      Input:            [208, 32, 124]
      Labels:           [13, -100, 124]
      Masked positions: [0 2]
      Changes:          pos0(13→[MASK]), pos2(124→kept)
      Loss targets:     2/3 positions

   📝 Example 2:
      Input:            [13, 180, 124]
      Labels:           [-100, 32, -100]
      Masked positions: [1]
      Changes:          pos1(32→180)
      Loss targets:     1/3 positions

   📝 Example 3:
      Input:            [13, 208, 124]
      Labels:           [-100, 32, -100]
      Masked positions: [1]
      Changes:          pos1(32→[MASK])
      Loss targets:     1/3 positions

🎭 Masking Examples for Caffeine:
   Original tokens: [13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
   Sequence length: 14

   📝 Example 1:
      Input:            [13, 180, 144, 159, 150, 150, 208, 54, 180, 150, 54, 180, 13, 13]
      Labels:           [-100, -100, -

# 6. Batch Processing

In [7]:

class MolecularBatchProcessor:
    """Handles batch processing of molecules for efficient training"""

    def __init__(self, vocabulary, mlm_processor, max_length=None):
        self.vocabulary = vocabulary
        self.mlm_processor = mlm_processor
        self.max_length = max_length  # If None, use dynamic batching
        self.pad_token_id = vocabulary.pad_token_id

    def process_molecule_batch(self, smiles_list):
        """Process a batch of SMILES strings into tokens"""
        batch_results = []
        valid_indices = []

        for i, smiles in enumerate(smiles_list):
            try:
                result = tokenize_molecule(smiles, self.vocabulary)
                batch_results.append(result)
                valid_indices.append(i)
            except (ValueError, RuntimeError) as e:
                print(f"⚠️ Skipping molecule {i}: {e}")
                continue

        return batch_results, valid_indices

    def create_padded_batch(self, token_sequences):
        """Convert variable-length sequences to padded tensors"""
        if not token_sequences:
            return torch.empty(0, 0, dtype=torch.long)

        # Find max length in batch
        batch_max_length = max(len(seq) for seq in token_sequences)

        # Apply global max length if specified
        if self.max_length:
            batch_max_length = min(batch_max_length, self.max_length)

        # Create padded batch tensor
        batch_size = len(token_sequences)
        batch_tensor = torch.full(
            (batch_size, batch_max_length), self.pad_token_id, dtype=torch.long
        )

        # Fill with actual sequences (truncate if needed)
        for i, seq in enumerate(token_sequences):
            seq_len = min(len(seq), batch_max_length)
            batch_tensor[i, :seq_len] = torch.tensor(seq[:seq_len], dtype=torch.long)

        return batch_tensor

    def create_attention_masks(self, batch_tensor):
        """Create attention masks (1 for real tokens, 0 for padding)"""
        return (batch_tensor != self.pad_token_id).long()

    def process_batch_for_mlm(self, smiles_batch, batch_size=None):
        """Complete pipeline: SMILES batch → MLM training tensors"""
        # Process molecules
        batch_results, valid_indices = self.process_molecule_batch(smiles_batch)

        if not batch_results:
            return None

        # Extract token sequences
        token_sequences = [result["tokens"] for result in batch_results]

        # Apply masking
        masked_samples = [
            self.mlm_processor.create_masked_sample(tokens)
            for tokens in token_sequences
        ]

        # Create input and label tensors
        input_sequences = [sample["input_tokens"] for sample in masked_samples]
        label_sequences = [sample["labels"] for sample in masked_samples]

        # Pad sequences
        input_tensor = self.create_padded_batch(input_sequences)
        label_tensor = self.create_padded_batch(label_sequences)

        # Create attention masks
        attention_mask = self.create_attention_masks(input_tensor)

        return {
            "input_ids": input_tensor,
            "labels": label_tensor,
            "attention_mask": attention_mask,
            "valid_indices": valid_indices,
            "batch_info": {
                "original_size": len(smiles_batch),
                "processed_size": len(batch_results),
                "max_sequence_length": input_tensor.shape[1],
            },
        }


# Create batch processor
batch_processor = MolecularBatchProcessor(vocabulary, mlm_processor, max_length=64)
print("🚀 Batch processor created!")
print(f"   Max sequence length: {batch_processor.max_length}")
print(f"   Pad token ID: {batch_processor.pad_token_id}")

🚀 Batch processor created!
   Max sequence length: 64
   Pad token ID: 0


In [18]:

def demonstrate_batch_processing():
    """Show batch processing with different scenarios"""

    # Test batch with mixed molecule sizes
    test_batch = [
        sample_molecules["Ethanol"],  # 3 atoms
        sample_molecules["Benzene"],  # 6 atoms
        sample_molecules["Glucose"],  # 12 atoms
        sample_molecules["Caffeine"],  # 14 atoms
        sample_molecules["Aspirin"],  # 13 atoms
        "INVALID_SMILES",  # Should be skipped
        sample_molecules["Morphine"],  # 21 atoms
    ]

    print("🎯 Batch Processing Demo")
    print("=" * 50)
    print(f"📦 Input batch size: {len(test_batch)}")
    print(
        f"📋 Molecules: {[name for name in ['Ethanol', 'Benzene', 'Glucose', 'Caffeine', 'Aspirin', 'INVALID', 'Morphine']]}"
    )

    # Process the batch
    batch_result = batch_processor.process_batch_for_mlm(test_batch)

    if batch_result is None:
        print("❌ Batch processing failed")
        return

    # Display results
    print(f"\n✅ Batch processed successfully!")
    print(f"   Original batch size: {batch_result['batch_info']['original_size']}")
    print(f"   Processed molecules: {batch_result['batch_info']['processed_size']}")
    print(f"   Valid indices: {batch_result['valid_indices']}")
    print(
        f"   Max sequence length: {batch_result['batch_info']['max_sequence_length']}"
    )

    print(f"\n📊 Tensor shapes:")
    print(f"   Input IDs: {batch_result['input_ids'].shape}")
    print(f"   Labels: {batch_result['labels'].shape}")
    print(f"   Attention mask: {batch_result['attention_mask'].shape}")

    # Show actual tensor content (first few molecules)
    print(f"\n🔍 Sample batch content (first 3 molecules):")
    for i in range(min(3, batch_result["input_ids"].shape[0])):
        input_seq = batch_result["input_ids"][i]
        label_seq = batch_result["labels"][i]
        attn_mask = batch_result["attention_mask"][i]

        # Find actual sequence length (non-padded)
        actual_length = attn_mask.sum().item()

        print(f"\n   Molecule {i} (length {actual_length}):")
        print(f"     Input:     {input_seq.tolist()}")
        print(f"     Labels:    {label_seq.tolist()}")
        print(f"     Attention: {attn_mask.tolist()}")

        # Show masking statistics
        masked_positions = (label_seq != -100).sum().item()
        print(
            f"     Masked: {masked_positions}/{actual_length} tokens ({masked_positions/actual_length:.1%})"
        )

    return batch_result


# Run the demonstration
demo_result = demonstrate_batch_processing()

🎯 Batch Processing Demo
📦 Input batch size: 7
📋 Molecules: ['Ethanol', 'Benzene', 'Glucose', 'Caffeine', 'Aspirin', 'INVALID', 'Morphine']
⚠️ Skipping molecule 5: Invalid SMILES string: INVALID_SMILES

✅ Batch processed successfully!
   Original batch size: 7
   Processed molecules: 6
   Valid indices: [0, 1, 2, 3, 4, 6]
   Max sequence length: 21

📊 Tensor shapes:
   Input IDs: torch.Size([6, 21])
   Labels: torch.Size([6, 21])
   Attention mask: torch.Size([6, 21])

🔍 Sample batch content (first 3 molecules):

   Molecule 0 (length 3):
     Input:     [13, 208, 124, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
     Labels:    [-100, 32, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
     Attention: [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
     Masked: 19/3 tokens (633.3%)

   Molecule 1 (length 6):
     Input:     [144, 208, 144, 144, 144, 144, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
     Labels:    [-100, 144, -100, -100, -100, -100,

[15:57:54] SMILES Parse Error: syntax error while parsing: INVALID_SMILES
[15:57:54] SMILES Parse Error: Failed parsing SMILES 'INVALID_SMILES' for input: 'INVALID_SMILES'


Theres seems to be an issue witht he batch processing. I would expect that all ipnut have the same length. they should be padded

In [14]:
# NEW CELL - Performance Analysis


def analyze_batch_efficiency():
    """Compare single vs batch processing performance"""
    import time

    print("⚡ Performance Analysis: Single vs Batch Processing")
    print("=" * 60)

    # Test molecules
    test_molecules = list(sample_molecules.values())

    # Single processing timing
    print("🐌 Single Processing:")
    start_time = time.time()
    single_results = []

    for smiles in test_molecules:
        try:
            # Tokenize
            tokens = tokenize_molecule(smiles, vocabulary)["tokens"]
            # Apply masking
            masked = mlm_processor.create_masked_sample(tokens)
            single_results.append(masked)
        except:
            continue

    single_time = time.time() - start_time
    print(f"   Time: {single_time:.4f}s for {len(single_results)} molecules")
    print(f"   Rate: {len(single_results)/single_time:.1f} molecules/second")

    # Batch processing timing
    print("\n🚀 Batch Processing:")
    start_time = time.time()

    batch_result = batch_processor.process_batch_for_mlm(test_molecules)

    batch_time = time.time() - start_time
    processed_count = (
        batch_result["batch_info"]["processed_size"] if batch_result else 0
    )

    print(f"   Time: {batch_time:.4f}s for {processed_count} molecules")
    if batch_time > 0:
        print(f"   Rate: {processed_count/batch_time:.1f} molecules/second")

    # Memory usage analysis
    if batch_result:
        print(f"\n💾 Memory Usage:")
        input_tensor = batch_result["input_ids"]
        total_elements = input_tensor.numel()
        memory_mb = total_elements * 8 / 1024 / 1024  # 8 bytes per int64

        print(f"   Tensor size: {input_tensor.shape}")
        print(f"   Total elements: {total_elements:,}")
        print(f"   Memory (approx): {memory_mb:.2f} MB")

        # Efficiency metrics
        non_pad_elements = batch_result["attention_mask"].sum().item()
        efficiency = non_pad_elements / total_elements
        print(f"   Padding efficiency: {efficiency:.1%}")
        print(f"   Wasted space: {(1-efficiency):.1%}")


# Run performance analysis
analyze_batch_efficiency()

⚡ Performance Analysis: Single vs Batch Processing
🐌 Single Processing:
   Time: 0.0015s for 8 molecules
   Rate: 5440.1 molecules/second

🚀 Batch Processing:
   Time: 0.0018s for 8 molecules
   Rate: 4342.5 molecules/second

💾 Memory Usage:
   Tensor size: torch.Size([8, 31])
   Total elements: 248
   Memory (approx): 0.00 MB
   Padding efficiency: 49.6%
   Wasted space: 50.4%


In [15]:
# NEW CELL - DataLoader Integration


class MolecularDataset(torch.utils.data.Dataset):
    """PyTorch Dataset for molecular data"""

    def __init__(self, smiles_list, batch_processor):
        self.smiles_list = smiles_list
        self.batch_processor = batch_processor
        # Pre-filter valid molecules for efficiency
        self.valid_smiles = []
        for smiles in smiles_list:
            try:
                tokenize_molecule(smiles, batch_processor.vocabulary)
                self.valid_smiles.append(smiles)
            except:
                continue

    def __len__(self):
        return len(self.valid_smiles)

    def __getitem__(self, idx):
        return self.valid_smiles[idx]


def collate_molecules(batch_smiles):
    """Custom collate function for DataLoader"""
    return batch_processor.process_batch_for_mlm(batch_smiles)


def demonstrate_dataloader():
    """Show how to use with PyTorch DataLoader"""
    print("📦 DataLoader Integration Demo")
    print("=" * 40)

    # Create larger dataset for demonstration
    extended_molecules = list(sample_molecules.values()) * 3  # 24 molecules

    # Create dataset
    dataset = MolecularDataset(extended_molecules, batch_processor)
    print(f"📊 Dataset created: {len(dataset)} valid molecules")

    # Create DataLoader
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=4,  # Small batch for demo
        shuffle=True,
        collate_fn=collate_molecules,
        drop_last=False,
    )

    print(f"🔄 DataLoader created: batch_size={dataloader.batch_size}")

    # Iterate through a few batches
    print(f"\n🎯 Processing batches:")
    for batch_idx, batch_data in enumerate(dataloader):
        if batch_data is None:
            print(f"   Batch {batch_idx}: Failed to process")
            continue

        print(
            f"   Batch {batch_idx}: {batch_data['input_ids'].shape} "
            + f"({batch_data['batch_info']['processed_size']} molecules)"
        )

        if batch_idx >= 2:  # Only show first 3 batches
            break

    print(f"\n✅ DataLoader integration successful!")
    print(f"   Total batches: {len(dataloader)}")
    print(f"   Ready for training loop!")


# Demonstrate DataLoader integration
demonstrate_dataloader()

📦 DataLoader Integration Demo
📊 Dataset created: 24 valid molecules
🔄 DataLoader created: batch_size=4

🎯 Processing batches:
   Batch 0: torch.Size([4, 31]) (4 molecules)
   Batch 1: torch.Size([4, 23]) (4 molecules)
   Batch 2: torch.Size([4, 31]) (4 molecules)

✅ DataLoader integration successful!
   Total batches: 6
   Ready for training loop!


## 📝 Updated Summary

This notebook demonstrated the **complete data construction pipeline** for MolE self-supervised pretraining with **batch processing capabilities**:

### ✅ Core Components:
1. **Molecular Tokenization**: SMILES → Morgan fingerprints → token sequences
2. **Vocabulary System**: Mapping atom environments to token IDs  
3. **Masked Language Modeling**: Random masking for self-supervision
4. **🚀 Batch Processing**: Efficient processing of multiple molecules
5. **📦 DataLoader Integration**: Ready for PyTorch training loops

### 🚀 New Batch Processing Features:
- **Variable-length padding**: Efficient tensor creation
- **Attention masks**: Proper handling of padded tokens
- **Error handling**: Graceful skipping of invalid molecules
- **Memory optimization**: Configurable max sequence lengths
- **Performance analysis**: Single vs batch processing comparison
- **DataLoader ready**: Direct integration with PyTorch training

### 🎯 Training Ready:
The batch processing pipeline now efficiently handles:
- Multiple molecules per batch (configurable batch size)
- Variable sequence lengths with proper padding
- Masked language modeling setup
- GPU-ready tensor formats
- Memory-efficient processing for large datasets

**Ready for training on 842M molecules!** 🚀

In [8]:
def analyze_morgan_fingerprints(smiles, molecule_name, max_radius=3):
    """
    Detailed analysis of Morgan fingerprints at different radii
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        print(f"❌ Invalid SMILES: {smiles}")
        return None
    
    print(f"🧬 Analyzing: {molecule_name}")
    print(f"   SMILES: {smiles}")
    print(f"   Formula: {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
    print(f"   Number of atoms: {mol.GetNumAtoms()}")
    print(f"   Number of bonds: {mol.GetNumBonds()}")
    
    # Analyze each radius
    for radius in range(max_radius + 1):
        print(f"\n📏 Radius {radius} Analysis:")
        
        # Get Morgan fingerprints with bitInfo
        info = {}
        fp = AllChem.GetMorganFingerprint(
            mol, 
            radius=radius, 
            bitInfo=info,
            includeRedundantEnvironments=True
        )
        
        print(f"   Total fingerprint bits: {len(info)}")
        
        # Analyze each atom's fingerprint
        atom_fingerprints = {}
        for atom_idx in range(mol.GetNumAtoms()):
            atom = mol.GetAtomWithIdx(atom_idx)
            
            # Find fingerprint for this atom at this radius
            atom_fingerprint = None
            for bit, atom_list in info.items():
                for atom_info_tuple in atom_list:
                    if atom_info_tuple[0] == atom_idx and atom_info_tuple[1] == radius:
                        atom_fingerprint = bit
                        break
                if atom_fingerprint is not None:
                    break
            
            if atom_fingerprint is not None:
                atom_fingerprints[atom_idx] = atom_fingerprint
                
        print(f"   Atom fingerprints found: {len(atom_fingerprints)}")
        
        # Show atom breakdown
        for atom_idx in sorted(atom_fingerprints.keys()):
            atom = mol.GetAtomWithIdx(atom_idx)
            fp_value = atom_fingerprints[atom_idx]
            # Check if this fingerprint is in vocabulary
            vocab_token_id = vocabulary.encode_atom_env(fp_value)
            print(f"     Atom {atom_idx:2d} ({atom.GetSymbol():2s}): FP={fp_value:>12} → Token={vocab_token_id:>3d}")
    
    return mol

# Analyze caffeine in detail
caffeine_mol = analyze_morgan_fingerprints(
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", 
    "Caffeine", 
    max_radius=2
)


🧬 Analyzing: Caffeine
   SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
   Formula: C8H10N4O2
   Number of atoms: 14
   Number of bonds: 15

📏 Radius 0 Analysis:
   Total fingerprint bits: 6
   Atom fingerprints found: 14
     Atom  0 (C ): FP=  2246728737 → Token= 13
     Atom  1 (N ): FP=  2092489639 → Token=180
     Atom  2 (C ): FP=  3218693969 → Token=144
     Atom  3 (N ): FP=  2041434490 → Token=159
     Atom  4 (C ): FP=  3217380708 → Token=150
     Atom  5 (C ): FP=  3217380708 → Token=150
     Atom  6 (C ): FP=  3217380708 → Token=150
     Atom  7 (O ): FP=   864942730 → Token= 54
     Atom  8 (N ): FP=  2092489639 → Token=180
     Atom  9 (C ): FP=  3217380708 → Token=150
     Atom 10 (O ): FP=   864942730 → Token= 54
     Atom 11 (N ): FP=  2092489639 → Token=180
     Atom 12 (C ): FP=  2246728737 → Token= 13
     Atom 13 (C ): FP=  2246728737 → Token= 13

📏 Radius 1 Analysis:
   Total fingerprint bits: 16
   Atom fingerprints found: 14
     Atom  0 (C ): FP=  3657471097 → Token=209


In [9]:
def analyze_vocabulary(vocabulary, sample_fingerprints=None):
    """
    Analyze the vocabulary structure and show sample mappings
    """
    if hasattr(vocabulary, 'vocab_dict'):
        vocab_dict = vocabulary.vocab_dict
        print(f"📚 Vocabulary Analysis:")
        print(f"   Total entries: {len(vocab_dict):,}")
        print(f"   Fingerprint range: {min(vocab_dict.keys()):,} to {max(vocab_dict.keys()):,}")
        print(f"   Token ID range: {min(vocab_dict.values())} to {max(vocab_dict.values())}")
        
        # Special tokens
        print(f"\n🎯 Special Tokens:")
        print(f"   PAD token ID: {vocabulary.pad_token_id}")
        print(f"   MASK token ID: {vocabulary.mask_token_id}")
        print(f"   UNK token ID: {vocabulary.unk_token_id}")
        
        # Show some sample mappings
        print(f"\n📝 Sample Fingerprint → Token Mappings:")
        sample_items = list(vocab_dict.items())[:10]
        for fingerprint, token_id in sample_items:
            print(f"   FP {fingerprint:>12} → Token {token_id:>3d}")
        
        if len(vocab_dict) > 10:
            print(f"   ... and {len(vocab_dict) - 10:,} more entries")
        
        # If we have specific fingerprints to check, show their mappings
        if sample_fingerprints:
            print(f"\n🔍 Specific Fingerprint Lookups:")
            for fp in sample_fingerprints:
                token_id = vocab_dict.get(fp, vocabulary.unk_token_id)
                status = "✅ Found" if fp in vocab_dict else "❌ Not found (→ UNK)"
                print(f"   FP {fp:>12} → Token {token_id:>3d} {status}")
    
    else:
        print("📚 Using simple vocabulary (structure not available for analysis)")

# Get some fingerprints from our caffeine analysis to check
caffeine_fingerprints = []
if caffeine_result:
    caffeine_fingerprints = [info['fingerprint'] for info in caffeine_result['atom_info'][:5]]

analyze_vocabulary(vocabulary, caffeine_fingerprints)


📚 Vocabulary Analysis:
   Total entries: 207
   Fingerprint range: 11,594,061 to 4,274,663,090
   Token ID range: 1 to 207

🎯 Special Tokens:
   PAD token ID: 0
   MASK token ID: 208
   UNK token ID: 209

📝 Sample Fingerprint → Token Mappings:
   FP   3387315712 → Token   1
   FP   2245273601 → Token   2
   FP    984188929 → Token   3
   FP   1016845826 → Token   4
   FP   1073491469 → Token   5
   FP   3156024848 → Token   6
   FP   1611362324 → Token   7
   FP   2246703125 → Token   8
   FP    881066519 → Token   9
   FP   1660341273 → Token  10
   ... and 197 more entries

🔍 Specific Fingerprint Lookups:
   FP   2246728737 → Token  13 ✅ Found
   FP   2092489639 → Token 180 ✅ Found
   FP   3218693969 → Token 144 ✅ Found
   FP   2041434490 → Token 159 ✅ Found
   FP   3217380708 → Token 150 ✅ Found


In [10]:
def create_training_tensors(molecule_data, device='cpu'):
    """
    Create PyTorch tensors from tokenized molecule data
    """
    if not molecule_data:
        return None
    
    tokens = molecule_data['tokens']
    
    print(f"🧮 Creating PyTorch Tensors:")
    print(f"   Molecule: {len(tokens)} atoms")
    print(f"   Device: {device}")
    
    # Create input tensor
    input_tensor = torch.tensor(tokens, dtype=torch.long, device=device)
    print(f"   Input tensor shape: {input_tensor.shape}")
    print(f"   Input tensor dtype: {input_tensor.dtype}")
    print(f"   Input tensor device: {input_tensor.device}")
    
    # Show tensor content
    print(f"   Tensor content: {input_tensor.tolist()}")
    
    # Create attention mask (all 1s since no padding in single molecule)
    attention_mask = torch.ones_like(input_tensor, dtype=torch.long, device=device)
    print(f"   Attention mask: {attention_mask.tolist()}")
    
    # Add batch dimension for model compatibility
    input_tensor_batched = input_tensor.unsqueeze(0)  # Shape: [1, seq_len]
    attention_mask_batched = attention_mask.unsqueeze(0)  # Shape: [1, seq_len]
    
    print(f"   Batched input shape: {input_tensor_batched.shape}")
    print(f"   Batched attention mask shape: {attention_mask_batched.shape}")
    
    return {
        'input_ids': input_tensor_batched,
        'attention_mask': attention_mask_batched,
        'raw_tokens': tokens,
        'sequence_length': len(tokens)
    }

# Create tensors for our molecules
print("🎯 Creating Training Tensors for Sample Molecules:")
print("=" * 60)

# Check CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if simple_test:
    print("\n📊 Ethanol:")
    ethanol_tensors = create_training_tensors(simple_test, device)

if caffeine_result:
    print("\n📊 Caffeine:")
    caffeine_tensors = create_training_tensors(caffeine_result, device)


🎯 Creating Training Tensors for Sample Molecules:
Using device: cuda

📊 Ethanol:
🧮 Creating PyTorch Tensors:
   Molecule: 3 atoms
   Device: cuda
   Input tensor shape: torch.Size([3])
   Input tensor dtype: torch.int64
   Input tensor device: cuda:0
   Tensor content: [13, 32, 124]
   Attention mask: [1, 1, 1]
   Batched input shape: torch.Size([1, 3])
   Batched attention mask shape: torch.Size([1, 3])

📊 Caffeine:
🧮 Creating PyTorch Tensors:
   Molecule: 14 atoms
   Device: cuda
   Input tensor shape: torch.Size([14])
   Input tensor dtype: torch.int64
   Input tensor device: cuda:0
   Tensor content: [13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
   Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   Batched input shape: torch.Size([1, 14])
   Batched attention mask shape: torch.Size([1, 14])


In [11]:
def compare_tokenization_radii(smiles, molecule_name, max_radius=3):
    """
    Compare tokenization results at different Morgan fingerprint radii
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        print(f"❌ Invalid SMILES: {smiles}")
        return None
    
    print(f"🔄 Radius Comparison for {molecule_name}:")
    print(f"   SMILES: {smiles}")
    print(f"   Atoms: {mol.GetNumAtoms()}")
    
    results = {}
    
    for radius in range(max_radius + 1):
        try:
            # Create modified tokenization function for this radius
            def tokenize_with_radius(smiles, vocab, r):
                mol = Chem.MolFromSmiles(smiles)
                if not mol:
                    return None
                
                tokens = []
                info = {}
                fp = AllChem.GetMorganFingerprint(
                    mol, radius=r, bitInfo=info, includeRedundantEnvironments=True
                )
                
                for atom_idx in range(mol.GetNumAtoms()):
                    atom_fingerprint = None
                    for bit, atom_list in info.items():
                        for atom_info_tuple in atom_list:
                            if atom_info_tuple[0] == atom_idx and atom_info_tuple[1] == r:
                                atom_fingerprint = bit
                                break
                        if atom_fingerprint is not None:
                            break
                    
                    if atom_fingerprint is None:
                        continue
                        
                    token_id = vocab.encode_atom_env(atom_fingerprint)
                    tokens.append(token_id)
                
                return tokens
            
            tokens = tokenize_with_radius(smiles, vocabulary, radius)
            
            if tokens:
                results[radius] = tokens
                unique_tokens = len(set(tokens))
                print(f"\n   📏 Radius {radius}:")
                print(f"      Tokens: {tokens}")
                print(f"      Length: {len(tokens)}")
                print(f"      Unique tokens: {unique_tokens}")
                print(f"      Vocabulary usage: {unique_tokens}/{len(tokens)} unique")
                
                # Token frequency
                from collections import Counter
                token_counts = Counter(tokens)
                most_common = token_counts.most_common(3)
                print(f"      Most frequent: {most_common}")
            else:
                print(f"\n   📏 Radius {radius}: ❌ No tokens generated")
                
        except Exception as e:
            print(f"\n   📏 Radius {radius}: ❌ Error: {e}")
    
    return results

# Compare different radii for caffeine
caffeine_radius_comparison = compare_tokenization_radii(
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Caffeine",
    max_radius=2
)


🔄 Radius Comparison for Caffeine:
   SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
   Atoms: 14

   📏 Radius 0:
      Tokens: [13, 180, 144, 159, 150, 150, 150, 54, 180, 150, 54, 180, 13, 13]
      Length: 14
      Unique tokens: 6
      Vocabulary usage: 6/14 unique
      Most frequent: [(150, 4), (13, 3), (180, 3)]

   📏 Radius 1:
      Tokens: [209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209]
      Length: 14
      Unique tokens: 1
      Vocabulary usage: 1/14 unique
      Most frequent: [(209, 14)]

   📏 Radius 2:
      Tokens: [209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209, 209]
      Length: 14
      Unique tokens: 1
      Vocabulary usage: 1/14 unique
      Most frequent: [(209, 14)]


In [12]:
def complete_pipeline_demo(smiles, molecule_name):
    """
    Demonstrate the complete pipeline from SMILES to training tensors
    """
    print(f"🚀 Complete Pipeline Demo: {molecule_name}")
    print(f"   Input SMILES: {smiles}")
    print("="*60)
    
    # Step 1: Tokenization
    print("\n📝 Step 1: Molecular Tokenization")
    try:
        tokenization_result = tokenize_molecule(smiles, vocabulary)
        print(f"   ✅ Success: {len(tokenization_result['tokens'])} tokens")
        print(f"   Token sequence: {tokenization_result['tokens']}")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        return None
    
    # Step 2: Masked Language Modeling
    print("\n🎭 Step 2: Masked Language Modeling")
    try:
        masked_sample = mlm_processor.create_masked_sample(tokenization_result['tokens'])
        print(f"   ✅ Success: {len(masked_sample['masked_positions'])} positions masked")
        print(f"   Input tokens:  {masked_sample['input_tokens']}")
        print(f"   Label tokens:  {masked_sample['labels']}")
        print(f"   Masked at:     {masked_sample['masked_positions']}")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        return None
    
    # Step 3: PyTorch Tensors
    print("\n🧮 Step 3: PyTorch Tensor Creation")
    try:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        # Input tensor
        input_tensor = torch.tensor(masked_sample['input_tokens'], dtype=torch.long, device=device)
        label_tensor = torch.tensor(masked_sample['labels'], dtype=torch.long, device=device)
        attention_mask = torch.ones_like(input_tensor, dtype=torch.long, device=device)
        
        # Add batch dimension
        batch_input = input_tensor.unsqueeze(0)
        batch_labels = label_tensor.unsqueeze(0)
        batch_attention = attention_mask.unsqueeze(0)
        
        print(f"   ✅ Success: Tensors created on {device}")
        print(f"   Input shape:     {batch_input.shape}")
        print(f"   Labels shape:    {batch_labels.shape}")
        print(f"   Attention shape: {batch_attention.shape}")
        print(f"   Memory usage:    {batch_input.numel() * 4} bytes (int32)")
        
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        return None
    
    # Step 4: Training readiness check
    print("\n🎯 Step 4: Training Readiness")
    
    # Check tensor properties
    checks = [
        ("Tensor device", batch_input.device.type == device),
        ("Consistent shapes", batch_input.shape == batch_attention.shape == batch_labels.shape),
        ("Valid token IDs", torch.all(batch_input >= 0) and torch.all(batch_input < len(vocabulary))),
        ("Valid label structure", torch.any(batch_labels != -100)),  # At least one label
        ("Proper masking", torch.any(batch_input == vocabulary.mask_token_id))  # Contains mask tokens
    ]
    
    all_passed = True
    for check_name, passed in checks:
        status = "✅" if passed else "❌"
        print(f"   {status} {check_name}")
        if not passed:
            all_passed = False
    
    if all_passed:
        print(f"\n🎉 Pipeline Complete! {molecule_name} is ready for training!")
    else:
        print(f"\n⚠️ Pipeline issues detected for {molecule_name}")
    
    return {
        'input_ids': batch_input,
        'labels': batch_labels,
        'attention_mask': batch_attention,
        'original_tokens': tokenization_result['tokens'],
        'masked_positions': masked_sample['masked_positions']
    }

# Run complete pipeline demo
demo_molecules = [
    ("Ethanol", "CCO"),
    ("Benzene", "C1=CC=CC=C1"),
    ("Caffeine", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
]

pipeline_results = {}
for name, smiles in demo_molecules:
    result = complete_pipeline_demo(smiles, name)
    if result:
        pipeline_results[name] = result
    print("\n" + "="*80 + "\n")


🚀 Complete Pipeline Demo: Ethanol
   Input SMILES: CCO

📝 Step 1: Molecular Tokenization
   ✅ Success: 3 tokens
   Token sequence: [13, 32, 124]

🎭 Step 2: Masked Language Modeling
   ✅ Success: 1 positions masked
   Input tokens:  [13, 208, 124]
   Label tokens:  [-100, 32, -100]
   Masked at:     [1]

🧮 Step 3: PyTorch Tensor Creation
   ✅ Success: Tensors created on cuda
   Input shape:     torch.Size([1, 3])
   Labels shape:    torch.Size([1, 3])
   Attention shape: torch.Size([1, 3])
   Memory usage:    12 bytes (int32)

🎯 Step 4: Training Readiness
   ✅ Tensor device
   ✅ Consistent shapes
   ✅ Valid token IDs
   ✅ Valid label structure
   ✅ Proper masking

🎉 Pipeline Complete! Ethanol is ready for training!


🚀 Complete Pipeline Demo: Benzene
   Input SMILES: C1=CC=CC=C1

📝 Step 1: Molecular Tokenization
   ✅ Success: 6 tokens
   Token sequence: [144, 144, 144, 144, 144, 144]

🎭 Step 2: Masked Language Modeling
   ✅ Success: 1 positions masked
   Input tokens:  [144, 208, 144, 1